# Diabetes Prediction using Lifestyle Data

Hi! In this notebook I am trying to predict whether a person has diabetes or not based on their lifestyle and health habits.

I am using a dataset from the CDC (BRFSS 2015 survey) which has answers from real people about things like their BMI, blood pressure, physical activity, etc.

**Goal:** Predict `Diabetes_binary` (0 = No diabetes, 1 = Diabetes)

## Importing Libraries

First I will import all the libraries that I need.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import pickle
import os

## 1. Loading the Data

Let me load the CSV file and see what the data looks like.

In [ ]:
df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')
print('Shape:', df.shape)
print('Missing values:', df.isnull().sum().sum())
df.head()

# checking how many people have diabetes vs dont have
print('Target distribution:')
print(df['Diabetes_binary'].value_counts())
print()
print('Non-diabetic:', (df['Diabetes_binary']==0).sum())
print('Diabetic:', (df['Diabetes_binary']==1).sum())

## 2. Selecting Features

The dataset has many columns but I only want to use the ones that a normal person can easily answer, like do you have high BP, do you smoke, what is your BMI etc.

I am picking these features:
- `HighBP` - High blood pressure
- `HighChol` - High cholesterol
- `BMI` - Body Mass Index
- `Smoker` - Do you smoke?
- `PhysActivity` - Are you physically active?
- `Fruits` / `Veggies` - Do you eat fruits and veggies?
- `HvyAlcoholConsump` - Heavy alcohol consumption
- `GenHlth` - General health rating
- `PhysHlth` - Physical health
- `DiffWalk` - Difficulty walking
- `Sex` - Gender
- `Age` - Age group

In [ ]:
# these are the features i am using
features = ['HighBP', 'HighChol', 'BMI', 'Smoker', 'PhysActivity',
            'Fruits', 'Veggies', 'HvyAlcoholConsump',
            'GenHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age']

target = 'Diabetes_binary'

print('Number of features:', len(features))
print()

# lets see which features are most related to diabetes
corr = df[features].corrwith(df[target]).abs().sort_values(ascending=False)
print('Correlation with diabetes:')
print(corr.round(3))

## 3. Exploratory Data Analysis (EDA)

Now I will make some graphs to understand the data better.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Diabetes - Lifestyle Feature Analysis', fontsize=14, fontweight='bold')

# plot 1 - how many diabetic vs non-diabetic
df[target].value_counts().plot(kind='bar', ax=axes[0,0],
    color=['steelblue','tomato'], edgecolor='black', rot=0)
axes[0,0].set_xticklabels(['No Diabetes','Diabetes'])
axes[0,0].set_title('Class Distribution')

# plot 2 - BMI distribution
df[df[target]==0]['BMI'].hist(bins=40, ax=axes[0,1], alpha=0.7, color='steelblue', label='No')
df[df[target]==1]['BMI'].hist(bins=40, ax=axes[0,1], alpha=0.7, color='tomato', label='Yes')
axes[0,1].set_title('BMI by Outcome')
axes[0,1].set_xlabel('BMI')
axes[0,1].legend()
axes[0,1].set_xlim(10, 70)

# plot 3 - Age distribution
df[df[target]==0]['Age'].hist(bins=13, ax=axes[0,2], alpha=0.7, color='steelblue', label='No')
df[df[target]==1]['Age'].hist(bins=13, ax=axes[0,2], alpha=0.7, color='tomato', label='Yes')
axes[0,2].set_title('Age Category by Outcome')
axes[0,2].legend()

# plot 4 - diabetes rate by general health
gen_rate = df.groupby('GenHlth')[target].mean()
axes[1,0].bar(gen_rate.index, gen_rate.values, color='tomato', edgecolor='k')
axes[1,0].set_title('Diabetes Rate by General Health')
axes[1,0].set_xlabel('General Health (1=Excellent, 5=Poor)')
axes[1,0].set_ylabel('Diabetes Rate')

# plot 5 - comparing lifestyle factors
binary_feats = ['HighBP','HighChol','Smoker','PhysActivity','DiffWalk','HvyAlcoholConsump']
diab_rates   = [df[df[target]==1][f].mean() for f in binary_feats]
nodiab_rates = [df[df[target]==0][f].mean() for f in binary_feats]
x = np.arange(len(binary_feats))
axes[1,1].bar(x-0.18, nodiab_rates, 0.35, label='No Diabetes', color='steelblue', edgecolor='k')
axes[1,1].bar(x+0.18, diab_rates,   0.35, label='Diabetes',    color='tomato',    edgecolor='k')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(binary_feats, rotation=25, ha='right', fontsize=8)
axes[1,1].set_title('Lifestyle Factor Rates by Outcome')
axes[1,1].legend()

# plot 6 - correlation bar chart
corr.sort_values().plot(kind='barh', ax=axes[1,2], color='steelblue', edgecolor='k')
axes[1,2].set_title('Feature Correlation with Diabetes')

plt.tight_layout()
plt.savefig('diabetes_eda.png', bbox_inches='tight')
plt.show()
print("EDA plot saved.")

## 4. Handling Class Imbalance

The data is very imbalanced - around 86% people dont have diabetes and only 14% have it. If I train the model on this directly it will just predict "no diabetes" most of the time.

So I will undersample the majority class (no diabetes) to make both classes equal.

In [ ]:
df_work = df[features + [target]].copy()

df_majority = df_work[df_work[target] == 0]
df_minority = df_work[df_work[target] == 1]

print('Before balancing:')
print('  No diabetes:', len(df_majority))
print('  Diabetes:', len(df_minority))

# undersample majority class
df_majority_down = resample(df_majority, replace=False,
                            n_samples=len(df_minority), random_state=42)
df_balanced = pd.concat([df_majority_down, df_minority]).sample(frac=1, random_state=42)

print()
print('After balancing:')
print(df_balanced[target].value_counts())

## 5. Train-Test Split

Now I will split the data into training set (80%) and testing set (20%). I am also scaling the features using StandardScaler so all values are on the same scale.

In [ ]:
X = df_balanced[features]
y = df_balanced[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

## 6. Training the Model

Now I will train a simple Logistic Regression model using the training data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# create and train the model
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_train_sc, y_train)

# test it
predictions = final_model.predict(X_test_sc)
accuracy = accuracy_score(y_test, predictions)

print(f'Model Accuracy: {accuracy * 100:.2f}%')

## 6. Saving the Model

Finally I will save the model, scaler, and feature list so that I can use them later in a web application.

In [ ]:
os.makedirs('models', exist_ok=True)

pickle.dump(final_model, open('models/diabetes_model.pkl',   'wb'))
pickle.dump(scaler,      open('models/diabetes_scaler.pkl',  'wb'))
pickle.dump(features,    open('models/diabetes_features.pkl','wb'))

print('Model saved successfully!')
print()
print('Features the web app will ask about:')
for f in features:
    print(' -', f)